# Notebook 05 — Mini-Network Training from Scratch
**Part 2: GymDetectorMini — < 10% of YOLOv7 parameters, no pretrained weights**  
**Team:** Varun Gazala · Mohit Raiyani · Jatinkumar Nabhoya · UNH Spring 2026

Run cells **top-to-bottom**. Cell 2 generates anchors (run once). Training uses the augmented dataset (648 images).

In [ ]:
# ── Config (single source of truth) ──────────────────────────────────────────
import os, sys
sys.path.insert(0, '..')   # make 'mininet' package importable

DATASET_DIR  = '../dataset_augmented'   # switch to '../dataset' for non-augmented run
WEIGHTS_OUT  = '../weights/mininet_best.pt'
RESULTS_OUT  = '../results/mininet_metrics.json'
ANCHORS_PATH = '../mininet/anchors.json'

IMG_SIZE  = 320
NC        = 5
NA        = 3
STRIDES   = [8, 16, 32]

# Training HPs
LR_INIT   = 1e-3
LR_MIN    = 1e-5
WD        = 5e-4
BATCH     = 16      # drop to 8 on CPU/Mac if OOM
EPOCHS    = 200
WARMUP_EP = 5       # linear LR warmup from LR_MIN → LR_INIT
PATIENCE  = 30      # early stop on val mAP@0.5
EMA_DECAY = 0.999

# Loss weights
LAMBDA_BOX = 0.05
LAMBDA_OBJ = 1.0
LAMBDA_CLS = 0.5
OBJ_POS_W  = 5.0

CLASS_NAMES = ['dumbbell', 'barbell', 'kettlebell', 'resistance_band', 'pull_up_bar']

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else \
         'mps'  if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}  |  PyTorch {torch.__version__}')
os.makedirs('../weights', exist_ok=True)
os.makedirs('../results', exist_ok=True)

In [ ]:
# ── Step 1: Anchor Generation (run once — saves anchors.json) ─────────────────
from mininet.anchors import generate_anchors, load_anchors

if not os.path.exists(ANCHORS_PATH):
    print('Running k-means anchor generation ...')
    ANCHORS = generate_anchors(
        labels_dir=f'{DATASET_DIR}/labels/train',
        n=9, img_size=IMG_SIZE, save=ANCHORS_PATH
    )
else:
    ANCHORS = load_anchors(ANCHORS_PATH)
    print(f'Loaded anchors from {ANCHORS_PATH}')
    for i, (g, s) in enumerate(zip(ANCHORS, STRIDES)):
        print(f'  P{3+i} (stride {s}): ' + ', '.join(f'({w:.1f}×{h:.1f})' for w,h in g))

In [ ]:
# ── Step 2: Build Model ───────────────────────────────────────────────────────
from mininet.model import GymDetectorMini

model = GymDetectorMini(nc=NC, na=NA).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
pct      = n_params / 37_200_000 * 100
print(f'Total parameters : {n_params:,}  ({pct:.2f}% of YOLOv7 base)')
assert n_params < 3_720_000, f'OVER BUDGET: {n_params:,} > 3,720,000'
print('✓ Within 10% parameter budget')

# Smoke-test forward pass
model.eval()
with torch.no_grad():
    outs = model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE))
for o, s in zip(outs, STRIDES):
    print(f'  stride {s:2d}  →  {tuple(o.shape)}')

In [ ]:
# ── Step 3: Datasets & DataLoaders ───────────────────────────────────────────
from torch.utils.data import DataLoader
from mininet.dataset  import GymMiniDataset, collate_fn
from mininet.augment  import TrainAugment

train_ds = GymMiniDataset(
    img_dir   = f'{DATASET_DIR}/images/train',
    label_dir = f'{DATASET_DIR}/labels/train',
    img_size  = IMG_SIZE,
    transform = TrainAugment(p_flip=0.5, p_jitter=0.5, p_affine=0.3),
)
val_ds = GymMiniDataset(
    img_dir   = f'{DATASET_DIR}/images/val',
    label_dir = f'{DATASET_DIR}/labels/val',
    img_size  = IMG_SIZE,
)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=0, collate_fn=collate_fn)

print(f'Train: {len(train_ds)} images  |  Val: {len(val_ds)} images')
print(f'Steps per epoch: {len(train_loader)}')

In [ ]:
# ── Step 4: Loss, Optimizer, Scheduler, EMA ──────────────────────────────────
from mininet.loss import YOLOLoss
from mininet.eval import EMA
import math

criterion = YOLOLoss(
    anchors_per_scale = ANCHORS,
    strides           = STRIDES,
    nc                = NC,
    img_size          = IMG_SIZE,
    lambda_box        = LAMBDA_BOX,
    lambda_obj        = LAMBDA_OBJ,
    lambda_cls        = LAMBDA_CLS,
    obj_pos_w         = OBJ_POS_W,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_INIT, weight_decay=WD)

# Cosine annealing after warmup
def lr_lambda(epoch):
    if epoch < WARMUP_EP:
        return (LR_MIN + (LR_INIT - LR_MIN) * epoch / WARMUP_EP) / LR_INIT
    t = (epoch - WARMUP_EP) / max(1, EPOCHS - WARMUP_EP)
    return (LR_MIN + 0.5 * (LR_INIT - LR_MIN) * (1 + math.cos(math.pi * t))) / LR_INIT

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
ema       = EMA(model, decay=EMA_DECAY)

print('Loss / optimizer / EMA ready.')

In [ ]:
# ── Step 5: Training Loop ─────────────────────────────────────────────────────
from mininet.eval import evaluate

history = {'train_loss': [], 'val_map50': [], 'lr': []}
best_map50  = 0.0
no_improve  = 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    ep_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE)
        optimizer.zero_grad()
        raw   = model(imgs)
        loss, _ = criterion(raw, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        ema.update(model)
        ep_loss += loss.item()

    scheduler.step()
    avg_loss = ep_loss / len(train_loader)

    # ── Val (every 5 epochs, always on epoch 1) ──
    if epoch == 1 or epoch % 5 == 0:
        ema.apply_shadow(model)
        val_res = evaluate(model, val_loader, ANCHORS, STRIDES,
                           conf_thres=0.001, iou_thres=0.6,
                           img_size=IMG_SIZE, device=DEVICE)
        ema.restore(model)
        map50 = val_res['map_50']
    else:
        map50 = history['val_map50'][-1] if history['val_map50'] else 0.0

    cur_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(avg_loss)
    history['val_map50'].append(map50)
    history['lr'].append(cur_lr)

    print(f'Epoch {epoch:3d}/{EPOCHS}  loss={avg_loss:.4f}  '
          f'val_mAP@0.5={map50:.4f}  lr={cur_lr:.2e}')

    # Save best checkpoint
    if map50 > best_map50:
        best_map50 = map50
        no_improve  = 0
        ema.apply_shadow(model)
        torch.save(model.state_dict(), WEIGHTS_OUT)
        ema.restore(model)
        print(f'  ↳ New best val mAP@0.5={best_map50:.4f} — saved {WEIGHTS_OUT}')
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f'Early stop at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

print(f'\nTraining complete. Best val mAP@0.5 = {best_map50:.4f}')

In [ ]:
# ── Step 6: Training Curves ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['val_map50'], color='tab:orange', label='Val mAP@0.5')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('mAP@0.5')
ax2.set_title('Validation mAP@0.5'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/figures/mininet_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/figures/mininet_loss_curves.png')

In [ ]:
# ── Step 7: Final Evaluation on Test Set ────────────────────────────────────
import json
from mininet.dataset import GymMiniDataset, collate_fn

# Load best checkpoint (EMA weights)
model.load_state_dict(torch.load(WEIGHTS_OUT, map_location=DEVICE))
model.eval()
print(f'Loaded best checkpoint: {WEIGHTS_OUT}')

test_ds = GymMiniDataset(
    img_dir   = f'{DATASET_DIR}/images/test',
    label_dir = f'{DATASET_DIR}/labels/test',
    img_size  = IMG_SIZE,
)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False,
                         num_workers=0, collate_fn=collate_fn)
print(f'Test set: {len(test_ds)} images')

test_res = evaluate(model, test_loader, ANCHORS, STRIDES,
                    conf_thres=0.001, iou_thres=0.6,
                    img_size=IMG_SIZE, device=DEVICE)

print('\n── Test Results ──────────────────────────────')
print(f'  mAP@0.5      : {test_res["map_50"]:.4f}')
print(f'  mAP@0.5:0.95 : {test_res["map"]:.4f}')
print('  Per-class AP@0.5:')
for cls, ap in test_res.get('per_class_ap50', {}).items():
    print(f'    {cls:<20} {ap:.4f}')

# Save metrics JSON (same schema as augmented_metrics.json)
out = {
    'model'          : 'GymDetectorMini (from scratch)',
    'dataset'        : DATASET_DIR,
    'n_params'       : n_params,
    'map_50'         : test_res['map_50'],
    'map'            : test_res['map'],
    'map_75'         : test_res['map_75'],
    'per_class_ap50' : test_res.get('per_class_ap50', {}),
    'training_history': history,
}
with open(RESULTS_OUT, 'w') as f:
    json.dump(out, f, indent=2)
print(f'\nResults saved → {RESULTS_OUT}')

In [ ]:
# ── Step 8: Visualize Predictions on Test Images ──────────────────────────────
import glob, random
import numpy as np
from mininet.utils  import decode_all, nms_single, draw_boxes_cv2
import matplotlib.pyplot as plt
import cv2

test_imgs = sorted(
    glob.glob(f'{DATASET_DIR}/images/test/*.jpg') +
    glob.glob(f'{DATASET_DIR}/images/test/*.jpeg') +
    glob.glob(f'{DATASET_DIR}/images/test/*.png')
)
show_paths = random.sample(test_imgs, min(6, len(test_imgs)))

model.eval()
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, path in zip(axes.flat, show_paths):
    img_bgr = cv2.imread(path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h0, w0  = img_rgb.shape[:2]

    from mininet.dataset import _letterbox
    img_lb, ratio, (pl, pt) = _letterbox(img_rgb, IMG_SIZE)
    img_t = torch.from_numpy(img_lb).permute(2,0,1).float().unsqueeze(0).to(DEVICE) / 255.0

    with torch.no_grad():
        raw  = model(img_t)
        pred = decode_all(raw, ANCHORS, STRIDES)[0]   # (N, 5+nc)
        dets = nms_single(pred, conf_thres=0.25, iou_thres=0.45)

    # Rescale boxes from letterbox coords → original image pixels
    if len(dets):
        d = dets.cpu().clone()
        d[:, 0] = (d[:, 0] - pl) / ratio
        d[:, 1] = (d[:, 1] - pt) / ratio
        d[:, 2] = (d[:, 2] - pl) / ratio
        d[:, 3] = (d[:, 3] - pt) / ratio
        d[:, :4] = d[:, :4].clamp(min=0)
        d[:, 0::2] = d[:, 0::2].clamp(max=w0)
        d[:, 1::2] = d[:, 1::2].clamp(max=h0)
        ann = draw_boxes_cv2(img_rgb, d.numpy().astype(int) if len(d) else d)
    else:
        ann = img_rgb

    ax.imshow(ann)
    ax.set_title(os.path.basename(path), fontsize=8)
    ax.axis('off')

plt.suptitle('Mini-Net — Test Set Predictions (conf≥0.25)', fontsize=13)
plt.tight_layout()
plt.savefig('../docs/figures/mininet_test_predictions.png', dpi=150, bbox_inches='tight')
plt.show()